# Week 5 Practice and Comprehensive Questions Solutions

In [ ]:
# Please load in advance as we will use later
# Data processing, processing and analysis library
import numpy as np
import numpy.random as random
import scipy as sp
from pandas import Series,DataFrame
import pandas as pd

# Visualization Library
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
%matplotlib inline

# Machine Learning Library
import sklearn

# Displayed to 3 decimal places
%precision 3

'%.3f'

####  <Practice Question 5-1>

(1)
We will use the car price data utilized in the main text. For this data, set the target variable as `price` and use `symboling`, `compression-ratio`, `stroke` as the explanatory variables to build a multiple regression model. When doing this, use `train_test_split` to split the data into training and test sets so that 80% of the whole data set is the training data. Then, construct the model and calculate the model's score using the test data. When executing `train_test_split`, set the `random_state` option to 0.

In [ ]:
# Acquire of automobile price data
import requests, zipfile
import io

url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
res = requests.get(url).content
auto = pd.read_csv(io.StringIO(res.decode('utf-8')), header=None)
auto.columns =['symboling', 'normalized-losses', 'make', 'fuel-type' , 'aspiration', 'num-of-doors',
                            'body-style', 'drive-wheels', 'engine-location', 'wheel-base', 'length', 'width', 'height',
                            'curb-weight', 'engine-type', 'num-of-cylinders', 'engine-size', 'fuel-system', 'bore',
                            'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'price']

In [ ]:
# Answer

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Set "price" as the target variable, and the others as explanatory variables
auto = auto[['price','symboling','compression-ratio','stroke']]
auto = auto.replace('?', np.nan).dropna()

# Specify "price" as the target variable and the others as explanatory variable
## WRITE ME
X = auto[['symboling','compression-ratio','stroke']]
y = auto['price']

# Split data into training and test sets
## WRITE ME
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Initialize and train the multiple regression class
## WRITE ME
model = LinearRegression()
model.fit(X_train, y_train)

# Display R^2 score
## WRITE ME
print('Coefficient of determination (train):{:.3f}'.format(model.score(X_train, y_train)))
print('Coefficient of determination (test):{:.3f}'.format(model.score(X_test, y_test)))

Coefficient of determination (train):0.004
Coefficient of determination (test):0.030


**Key Takeaway**: If you did this right, you should have a very low score for both the training and testing data compared to when we used `horsepower`, `width`, `height` in the main text.. This is because the variables we used are not very good metrics for pricing cars:

*   `symboling`: This is an insurance risk rating, not a pricing metric like market value or production cost
*   `compression-ratio` and `stroke`: These are niche engine mechanics that show very weak linear correlation with the total vehicle price

As you can see, variables selection matters. Including numerical variables simply because they are available will not help your model if they lack real correlation with the target. You will see this many times throughout the course, including in the Competition and the Final Assignment.

(2) Now, use the same target variable, `price`, but construct a multiple regression model using a different set of explanatory variables. Split the data so that 80% is used for training and 20% for testing, then calculate the model's score using the test data. When splitting the data into training and test sets, set the `random_state` function to 0. How has the model's result changed? Analyze and discuss the reasons for the change.

In [ ]:
# @title
# Retrieve car price data
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
res = requests.get(url).content

auto = pd.read_csv(io.StringIO(res.decode('utf-8')), header=None)

auto.columns = [
    'symboling','normalized-losses','make','fuel-type' ,'aspiration','num-of-doors',
    'body-style','drive-wheels','engine-location','wheel-base','length','width','height',
    'curb-weight','engine-type','num-of-cylinders','engine-size','fuel-system','bore',
    'stroke','compression-ratio','horsepower','peak-rpm','city-mpg','highway-mpg','price'
]

In [ ]:
# Answer

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Set "price" as the target variable, and the others as explanatory variables
auto = auto[['price','length','width','height']]
auto = auto.replace('?', np.nan).dropna()

# Specify "price" as the target variable and the others as explanatory variable
## WRITE ME
X = auto[['length','width','height']]
y = auto['price']

# Split data into training and test sets
## WRITE ME
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Initialize and train the multiple regression class
## WRITE ME
model = LinearRegression()
model.fit(X_train, y_train)

# Display R^2 score
## WRITE ME
print('Coefficient of determination (train):{:.3f}'.format(model.score(X_train, y_train)))
print('Coefficient of determination (test):{:.3f}'.format(model.score(X_test, y_test)))

Coefficient of determination (train):0.657
Coefficient of determination (test):0.393


# Comments
1. Model performance change
- If R² improves → selected features better explain price
- If R² drops → features are less informative or noisy
- If train ≫ test → overfitting
- If both low → underfitting

2. Feature selection
- Strong predictors like: engine-size, horsepower, curb-weight, usually improve performance
- Weak or irrelevant features → reduce accuracy

#### <Practice Question 5-2>

You are the Lead Data Scientist at a national bank. The hardware team is upgrading ATMs with new optical scanners to detext counterfeit bills based on 4 image metrics (`variance`,`skewness`,`kurtosis`, and`entropy`). Your job is to test different machine learning approaches and recommend the best one for the ATM software.

In this question, we will develop a baseline model. In order to ensure explainability, we will utilize what we just learned and use our knowledge about the Logistic Regression model.

In [ ]:
# Setup the data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
df_counterfeit = pd.read_csv(url, names=['variance', 'skewness', 'kurtosis', 'entropy', 'is_fake'])
df_counterfeit.head()

# Output the data's shape and missing values
print('Data Format:{}'.format(df_counterfeit.shape))
print('Number of defects:{}'.format(df_counterfeit.isnull().sum().sum()))

# Output the first 5 rows of the data
df_counterfeit.head()

Data Format:(1372, 5)
Number of defects:0


,variance,skewness,kurtosis,entropy,is_fake
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0


(1)
Train a Logistic Regression model on the **scaled** training data and print the test accuracy. Use 80% of the data for training and the other 20% for testing. Make sure to set `random_state=0`.

In [ ]:
# Answer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# (1)
# Split into training and testing data
## WRITE ME
X = df_counterfeit.drop('is_fake',axis=1)
y = df_counterfeit['is_fake']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

# Standardize the data
## WRITE ME
sc = StandardScaler()
sc.fit(X_train)
X_train_std = sc.transform(X_train)
X_test_std = sc.transform(X_test)

# Build the Logistic Regression model and print the accuracy
## WRITE ME
lr_model = LogisticRegression()
lr_model.fit(X_train_std,y_train)
print(f"Train data accuracy: {lr_model.score(X_train_std,y_train):.3f}")
print(f"Test data accuracy: {lr_model.score(X_test_std,y_test):.3f}")


Train data accuracy: 0.980
Test data accuracy: 0.989


(2) Print the model's coeficients using `model.coef_`.

In [ ]:
# (2)
# WRITE ME
print(lr_model.coef_)

[[-4.691 -4.76  -4.31   0.14 ]]


# Comments
1. Standardization (Z-score scaling)

Transforms data to have mean = 0 and standard deviation = 1

Characteristics:
- Centered around 0
- Values are not bounded (can be negative or >1)
- Preserves outliers (doesn’t squash them)

When to use:
- Models assuming Gaussian-like data
- Linear models (Linear/Logistic Regression)
- SVM, PCA


2. Normalization (Min-Max scaling)

Scales data to a fixed range, usually [0, 1]
Characteristics:
- Bounded range (e.g., 0 to 1)
- Sensitive to outliers (they stretch the scale)
- Keeps relative shape of distribution

When to use:
- Neural networks (especially with bounded activations)
- Distance-based models (KNN, clustering)
- When you need fixed range inputs

####  <Practice Question 5-3>

Going back to your role of being a Lead Data Scientist at a national bank, the engineering team pointed out that calculating complex math equations on the ATM's basic microchip is slow, so your previous Logistic Regression model is not usable in ATMs. Next, we are going to use a Decision Tree model instead, since all it is doing is creating simple "IF/THEN" rules.



In [ ]:
# Setup the data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
df_counterfeit = pd.read_csv(url, names=['variance', 'skewness', 'kurtosis', 'entropy', 'is_fake'])
df_counterfeit.head()

# Output the data's shape and missing values
print('Data Format:{}'.format(df_counterfeit.shape))
print('Number of defects:{}'.format(df_counterfeit.isnull().sum().sum()))

# Output the first 5 rows of the data
df_counterfeit.head()

Data Format:(1372, 5)
Number of defects:0


,variance,skewness,kurtosis,entropy,is_fake
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0


(1) Train a default Decision Tree model on the **unscaled** training data (remember, trees do not care about scaling) using the default parameter for `max_depth` and print the test accuracy. Use 80% of the data for training and the other 20% for testing. Make sure to set `random_state=0`.

In [ ]:
# Answer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# (1)
# Split into training and testing data
## WRITE ME
X = df_counterfeit.drop('is_fake',axis=1)
y = df_counterfeit['is_fake']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=0)

# Build the Decision Tree model and print the accuracy
## WRITE ME
dt_model = DecisionTreeClassifier(criterion='entropy',random_state=0)
dt_model.fit(X_train,y_train)
print('Accuracy(train):{:.3f}'.format(dt_model.score(X_train, y_train)))
print('Accuracy(test):{:.3f}'.format(dt_model.score(X_test, y_test)))

Accuracy(train):1.000
Accuracy(test):0.985


(2) The engineering team reviewed your default Decision Tree mode. Because it has no depth limit, it grew deep, creating a web of rules to achieve high accuracy. The ATM's microchip processes logic very slowly and can only handle a maximum of 4 consecutive "IF/THEN" rules before the transaction times out.

Next, we are going to test if this new contraint ruins the model's ability to catch fake bills. Do the same thing you did in (1), but instead set `max_depth=4` and compare the accuracy scores.

In [ ]:
# (2)
# Build the Decision Tree model with the specified parameters and print the accuracy
## WRITE ME
dt_model = DecisionTreeClassifier(criterion='entropy',random_state=0,max_depth=4)
dt_model.fit(X_train,y_train)
print('Accuracy(train):{:.3f}'.format(dt_model.score(X_train, y_train)))
print('Accuracy(test):{:.3f}'.format(dt_model.score(X_test, y_test)))

Accuracy(train):0.987
Accuracy(test):0.967


# Comments
Effect of max_depth

Depth = 3
- Train score: moderate
- Test score: often good
- → ✅ Good generalization (simple model)

Depth = 5
- Train score: higher
- Test score: similar or slightly better
- → ✅ Often best balance

Depth = 10
- Train score: very high (often ~1.0)
- Test score: may drop
- → ⚠️ Overfitting

Key idea:
- As depth increases → model memorizes training data → generalization worsens

####  <Practice Question 5-4>

Ridge and Lasso Regression perform better than normal Multiple Regression when one or more of these characteristcs is true for the dataset:
1.  There are not many data samples compared to the number of features
2.  There is strong colinearity amongst the features

Let's observe this phenomenon. Run the cell below to download the diabetes dataset, where the objective variable is a quantitative measure of diabetes progression.

In [ ]:
from sklearn.datasets import load_diabetes
X, y = load_diabetes(return_X_y=True)

Split the data into training and testing data, and train a LinearRegression, Ridge, and Lasso model on the scaled data. However, in order to observe the LinearRegression's low performance compared to Ridge and Lasso in unideal conditions, set `train_size=30` and `random_state=0`. This means that we are only going to be using 30 out of 442 data points for training, causing overfitting.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Train-test split with train_size=30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=30,random_state=0
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit models
lr = LinearRegression()
ridge = Ridge()
lasso = Lasso()

for model in [lr, ridge,lasso]:
    model.fit(X_train, y_train)
    print('{} (train):{:.6f}'.format(model.__class__.__name__ , model.score(X_train, y_train)))
    print('{} (test):{:.6f}'.format(model.__class__.__name__ , model.score(X_test, y_test)))

LinearRegression (train):0.705834
LinearRegression (test):0.312854
Ridge (train):0.207768
Ridge (test):0.187724
Lasso (train):0.541103
Lasso (test):0.429662


####  <Practice Question 5-5>

The operation team at the national bank you work in suspects that counterfeiters target high-traffic ATMs. To test this, they merged a new column into your image scanner dataset: `volume` (the number of transactions that ATM processes that day, ranging from 10,000 to 80,000).

A junior analyst immediately runs a k-Nearest Neighbor model on this new combined dataset.
The accuracy plumets. You need to show the analyst why distance-based models fail when you mix massive business metrics with tiny scientific metrics.

The code below simulates the operations team adding the massive `volume` metric.



In [ ]:
#Setup the data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
df_counterfeit = pd.read_csv(url, names=['variance', 'skewness', 'kurtosis', 'entropy', 'is_fake'])

X = df_counterfeit.drop('is_fake',axis=1)
y = df_counterfeit['is_fake']

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.8, random_state=0)

# Create simulated volume column
np.random.seed(0)
X_train_merged = X_train.copy()
X_test_merged = X_test.copy()
X_train_merged['volume'] = np.random.randint(10000, 80000, size=len(X_train))
X_test_merged['volume'] = np.random.randint(10000, 80000, size=len(X_test))
X_test_merged.head()

,variance,skewness,kurtosis,entropy,volume
1023,-1.7713,-10.766500,10.21840,-1.00430,72444
642,5.1321,-0.031048,0.32616,1.11510,77889
1196,-2.0149,3.687400,-1.93850,-3.89180,17734
31,1.4884,3.627400,3.30800,0.48921,38436
253,5.2868,3.257000,-1.37210,1.16680,33905


(1) Train an unscaled `KNeighborsClassifier` on this combined data and record the accuracy on the test data. For simplicity, use `n_neighbors=5` (no need to find the optimal value).

In [ ]:
# Answer
from sklearn.neighbors import  KNeighborsClassifier
from sklearn.model_selection import train_test_split

# (1)
# WRITE ME
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train_merged,y_train)
unscaled_score = knn_unscaled.score(X_test_merged,y_test)
print(f"k-NN (Unscaled Merged Data) Accuracy: {unscaled_score:.3f}")

k-NN (Unscaled Merged Data) Accuracy: 0.500


(2) Standardize the combined data, and train a second `KNeighborsClassifier` on the data and record the accuracy. Then compare it with the score you got in (1).

In [ ]:
# (2)

# Standardize the combined data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_merged)
X_test_scaled = scaler.transform(X_test_merged)

# Create k-NN model and print accuracy
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
scaled_score = knn_scaled.score(X_test_scaled, y_test)
print(f"k-NN (Scaled Merged Data) Accuracy: {scaled_score:.3f}")

k-NN (Scaled Merged Data) Accuracy: 0.974


####  <Practice Question 5-6>

The Cheif Risk Officer (CRO) is highly satisfied with the accuracies, but wants the human security team to manually study the most sophisticated counterfeit bills-the ones that almost tricked the ML algorithm. Instead of forcing the team to review all 1,372 bills, you can use a Support Vector Machine. The strength of the SVM algorithm lies in that they explicitly identify the exact data points (or we call them the "support vectors") that sit close to the boundary between counterfeit and real. We are going to point these out so that the everybody knows that kind of bills they have to be cautious of.



In [ ]:
#Setup the data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
df_counterfeit = pd.read_csv(url, names=['variance', 'skewness', 'kurtosis', 'entropy', 'is_fake'])

X = df_counterfeit.drop('is_fake',axis=1)
y = df_counterfeit['is_fake']

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

(1) Train a `SVC` (with default parameters) on the **scaled** training data and
print the test accuracy.

In [ ]:
# Answer
from sklearn.svm import LinearSVC

# (1)
# Standardize the data
## WRITE ME
sc = StandardScaler()
sc.fit(X_train)
X_train_std = sc.transform(X_train)
X_test_std = sc.transform(X_test)

# Create a SVC model and print accuracy
svm_model = LinearSVC()
svm_model.fit(X_train_std, y_train)
svm_score = svm_model.score(X_test_std, y_test)
print(f"SVM Test Accuracy: {svm_score:.3f}")

SVM Test Accuracy: 0.980


(2) The `LinearSVC` class has a method `.decision_function()` that calculates the mathematical distance of each bill from the boundary line. You are going to use this by calling
```python
model.decision_function(data)
```
where `model` is the name you give the SVM, and `data` is the data you want to calculate the distance of. Count the number of bills that fall within a distance of 1.0 from the boundary.

In [ ]:
# (2)
# Determine the distance for all of the testing data
## WRITE ME
distances = svm_model.decision_function(X_test_std)

# Count the number of data points that fall within a distance of 1.0 from the boundary and print the cout
## WRITE ME
danger_count = np.sum(np.abs(distances) < 1.0)
print(f"Bills flagged in the Danger Zone: {danger_count}")

Bills flagged in the Danger Zone: 156


## 5.9 Comprehensive Question

### ■ Comprehensive Question 5-1 No Free Lunch

So far, we've worked with various datasets, such as math test scores and breast cancer data. Using these datasets, try applying the models we've covered—such as logistic regression and SVM—and check which model achieves the highest score. You'll find that the model with the best score varies depending on the data. Consider the characteristics of the models that perform best for each dataset. This concept is known as the "No Free Lunch" theorem, meaning that no single model is universally the best for all types of data.

In [ ]:
# Answer
# Load the necessary libraries
from sklearn.neighbors import  KNeighborsClassifier
from sklearn.tree import  DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC

# Use breast cancer data loaded with load_breast_cancer as an example
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify = cancer.target, random_state=0)

# Standardization
sc = StandardScaler()
sc.fit(X_train)
X_train_std = sc.transform(X_train)
X_test_std = sc.transform(X_test)

# Multiple model configurations
models = {
    'knn':  KNeighborsClassifier(),
    'tree': DecisionTreeClassifier(random_state=0),
    'logistic': LogisticRegression(random_state=0),
    'svc1': LinearSVC(random_state=0),
    'svc2': SVC(random_state=0)
}

# Empty dictionary data to have scores
scores = {}

# Scores are calculated for each model.
for model_name, model in models.items():
    model.fit(X_train_std, y_train)
    scores[(model_name, 'train')] = model.score(X_train_std, y_train)
    scores[(model_name, 'test')] = model.score(X_test_std, y_test)

# Finally, each score result is displayed
pd.Series(scores).unstack()

,test,train
knn,0.951049,0.978873
logistic,0.958042,0.990610
svc1,0.951049,0.992958
svc2,0.958042,0.992958
tree,0.902098,1.000000




---



**Acknowledgements**: Regarding the use of the following four datasets
1. http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data
2. http://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
3. http://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data
4. https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt

**References**: Dua, D. and Graff, C. (2019). UCI Machine Learning Repository [[http://archive.ics.uci.edu/ml](http://archive.ics.uci.edu/ml)]. Irvine, CA: University of California, School of Information and Computer Science.

- Note on citation of dataset in 3:  
P. Cortez and A. Silva. Using Data Mining to Predict Secondary School Student Performance. In A. Brito and J. Teixeira Eds., Proceedings of 5th FUture BUsiness TEChnology Conference (FUBUTEC 2008) pp.5-12, Porto, Portugal, April, 2008, EUROSIS, ISBN 978-9077381-39-7.
[http://www3.dsi.uminho.pt/pcortez/student.pdf](http://www3.dsi.uminho.pt/pcortez/student.pdf)  